In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import io
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from backtest.data import DataPanel, FieldSpec  # engine data layer (used from Cell 4 on)

# --- HF Data Library API config ---------------------------------------------
# Key is read from the environment (set HFD_API_KEY before launching the kernel,
# or uncomment the line below to set it inline for this session only).
# os.environ["HFD_API_KEY"] = "hfd_..."
HFD_API_KEY = os.environ.get("HFD_API_KEY", "")
if not HFD_API_KEY and Path(".hf_api_key").exists():
    HFD_API_KEY = Path(".hf_api_key").read_text().strip()
if not HFD_API_KEY:
    raise RuntimeError(
        "HFD_API_KEY not set. Set the env var (recommended) or uncomment the "
        "os.environ line above with your hfd_... key, then re-run."
    )

HFD_BASE = "https://api.hfdatalibrary.com/v1"
CACHE_DIR = Path("data/hf_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

_session = requests.Session()
_session.headers.update({"X-API-Key": HFD_API_KEY})


def hf_fetch(ticker: str, timeframe: str = "1min", version: str = "clean") -> pd.DataFrame:
    """Cache-first fetch from HF Data Library (returns full per-ticker history).

    timeframe="1min" -> GET /bars/{ticker}                 (direct, ~30 MB parquet)
    timeframe in {"5min","daily",...} -> two-step signed-URL flow:
        GET /download-token/{ticker}?timeframe=... -> {url} -> GET url (parquet)
    The API returns the FULL history per ticker regardless of date params, so we
    download once, cache to parquet on disk, and slice the window we need offline.
    """
    cache_fp = CACHE_DIR / f"{ticker}_{timeframe}.parquet"
    if cache_fp.exists():
        return pd.read_parquet(cache_fp)
    if timeframe == "1min":
        r = _session.get(f"{HFD_BASE}/bars/{ticker}", params={"version": version}, timeout=300)
        r.raise_for_status()
        content = r.content
    else:
        tok = _session.get(
            f"{HFD_BASE}/download-token/{ticker}",
            params={"version": version, "timeframe": timeframe, "format": "parquet"},
            timeout=60,
        )
        tok.raise_for_status()
        signed_url = tok.json()["url"]
        dl = requests.get(signed_url, timeout=300)  # token authorizes; no key header
        dl.raise_for_status()
        content = dl.content
    df = pd.read_parquet(io.BytesIO(content))
    df.to_parquet(cache_fp)
    return df


def hf_symbols() -> pd.DataFrame:
    """List all available tickers (cached)."""
    cache_fp = CACHE_DIR / "symbols.json"
    if cache_fp.exists():
        payload = json.loads(cache_fp.read_text())
    else:
        r = _session.get(f"{HFD_BASE}/symbols", timeout=60)
        r.raise_for_status()
        payload = r.json()
        cache_fp.write_text(json.dumps(payload))
    return pd.DataFrame(payload["symbols"])


# --- connectivity + daily-schema probe --------------------------------------
symbols = hf_symbols()
print(f"Symbols available: {len(symbols)}")

probe_tickers = ["AAPL", "MSFT", "SPY", "F", "AMD"]
probe = {t: hf_fetch(t, timeframe="daily") for t in probe_tickers}

aapl = probe["AAPL"]
print(f"\nAAPL daily — shape={aapl.shape}, cols={list(aapl.columns)}")
print(f"date range: {aapl['datetime'].min()} -> {aapl['datetime'].max()}")
print("\nper-ticker daily rows:")
for t, d in probe.items():
    print(f"  {t:5s} rows={len(d):6d}  {d['datetime'].min().date()} -> {d['datetime'].max().date()}")

aapl.head(3)

Symbols available: 1391



AAPL daily — shape=(5909, 7), cols=['datetime', 'Open', 'High', 'Low', 'Close', 'Volume', 'source']
date range: 2002-12-30 00:00:00 -> 2026-06-26 00:00:00

per-ticker daily rows:
  AAPL  rows=  5909  2002-12-30 -> 2026-06-26
  MSFT  rows=  5909  2002-12-30 -> 2026-06-26
  SPY   rows=  5908  2002-12-30 -> 2026-06-26
  F     rows=  5908  2002-12-30 -> 2026-06-26
  AMD   rows=  5909  2002-12-30 -> 2026-06-26


,datetime,Open,High,Low,Close,Volume,source
0,2002-12-30,0.23,0.23,0.23,0.23,144444272,pitrading
1,2002-12-31,0.23,0.24,0.23,0.24,182653296,pitrading
2,2003-01-02,0.24,0.24,0.24,0.24,169439816,pitrading


In [2]:
# =====================================================================
# Cell 2 — Point-in-time universe (spec AddUniverse stage)
#   eligibility @ day t (using info through t-1):  price > $5,  exclude SPY,
#   rank by previous-day dollar volume, keep top N.
#   (ATR > $0.50, RelVol > 1, top-20 by RelVol come later, in the strategy.)
# =====================================================================
WINDOW_START = "2016-01-01"
WINDOW_END   = "2023-12-31"
UNIVERSE_N   = 1000            # spec universe_size
MIN_PRICE    = 5.0             # spec min_price
EXCLUDE      = {"SPY"}         # spec: f.Symbol != spy
WARMUP_DAYS  = 40              # calendar-day buffer before WINDOW_START for ATR/RelVol warmup

TICKERS = sorted(symbols["ticker"].astype(str).tolist())
print(f"Catalogue tickers: {len(TICKERS)}")


def load_etf_symbols():
    """Authoritative ETF flag from the NASDAQ symbol directory (cached)."""
    fp = CACHE_DIR / "nasdaqtraded.txt"
    if not fp.exists():
        r = requests.get("https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt", timeout=60)
        r.raise_for_status()
        fp.write_bytes(r.content)
    ref = pd.read_csv(fp, sep="|")
    ref = ref[ref["Symbol"].notna() & ref["ETF"].notna()]  # drop trailing footer row
    return set(ref.loc[ref["ETF"] == "Y", "Symbol"].astype(str))

etf_symbols = load_etf_symbols()
print(f"ETF symbols in NASDAQ directory: {len(etf_symbols)}")


def fetch_daily_safe(ticker: str, retries: int = 4):
    """hf_fetch(daily) with 429 backoff; returns None on 404/permanent failure."""
    for k in range(retries):
        try:
            return hf_fetch(ticker, timeframe="daily")
        except requests.HTTPError as e:
            code = getattr(e.response, "status_code", None)
            if code == 429:
                time.sleep(2.0 * (k + 1)); continue
            if code == 404:
                return None
            if k == retries - 1:
                return None
            time.sleep(1.0)
        except Exception:
            if k == retries - 1:
                return None
            time.sleep(1.0)
    return None


# --- fetch daily for the full catalogue (cache-first) ---
daily_frames, failures = {}, []
for i, t in enumerate(TICKERS):
    d = fetch_daily_safe(t)
    if d is None or len(d) == 0:
        failures.append(t)
    else:
        daily_frames[t] = d
    if not (CACHE_DIR / f"{t}_daily.parquet").exists():
        time.sleep(0.4)  # throttle only on real network pulls (<=300 req/min)
    if (i + 1) % 200 == 0:
        print(f"  fetched {i+1}/{len(TICKERS)}  (failures so far: {len(failures)})")

print(f"Daily frames: {len(daily_frames)}  |  failures: {len(failures)}")

# --- assemble daily close & volume panels ---
closes = {t: d.set_index("datetime")["Close"] for t, d in daily_frames.items()}
vols   = {t: d.set_index("datetime")["Volume"] for t, d in daily_frames.items()}
close  = pd.DataFrame(closes).sort_index()
volume = pd.DataFrame(vols).reindex_like(close)

# restrict to window + warmup buffer
buf_start = pd.Timestamp(WINDOW_START) - pd.Timedelta(days=WARMUP_DAYS)
in_range  = (close.index >= buf_start) & (close.index <= pd.Timestamp(WINDOW_END))
close, volume = close.loc[in_range], volume.loc[in_range]

dollar_vol = close * volume

# --- PIT eligibility (use info through t-1 via shift) ---
px_lag = close.shift(1)
dv_lag = dollar_vol.shift(1)
eligible = (px_lag > MIN_PRICE) & px_lag.notna()
etfs_in_set = set(eligible.columns) & etf_symbols
for s in (EXCLUDE | etfs_in_set):
    if s in eligible.columns:
        eligible[s] = False
print(f"ETFs removed from candidate set: {len(etfs_in_set)}  (e.g. {sorted(etfs_in_set)[:8]})")

# rank eligible names by previous-day dollar volume, keep top N
dv_ranked = dv_lag.where(eligible)
rank = dv_ranked.rank(axis=1, ascending=False, method="first")
universe = (rank <= UNIVERSE_N) & eligible

# --- inspect (over the tradable window only) ---
trade_mask = universe.index >= pd.Timestamp(WINDOW_START)
members_per_day = universe.loc[trade_mask].sum(axis=1)
print(f"\nclose panel: {close.shape}  {close.index.min().date()} -> {close.index.max().date()}")
print(f"universe:    {universe.shape}")
print(f"members/day over {WINDOW_START}..{WINDOW_END}: "
      f"mean={members_per_day.mean():.0f}  min={members_per_day.min()}  max={members_per_day.max()}")
print(f"distinct names ever in universe: {int((universe.loc[trade_mask].sum(axis=0) > 0).sum())}")

sample_day = members_per_day.index[-1]
top_today = dv_ranked.loc[sample_day].dropna().sort_values(ascending=False).head(10)
print(f"\nTop-10 by prev-day $vol on {sample_day.date()}:")
print((top_today / 1e9).round(3).rename("prev_day_$vol_bn"))

Catalogue tickers: 1391
ETF symbols in NASDAQ directory: 5419


  fetched 200/1391  (failures so far: 2)


  fetched 400/1391  (failures so far: 2)


  fetched 600/1391  (failures so far: 2)


  fetched 800/1391  (failures so far: 2)


  fetched 1000/1391  (failures so far: 3)


  fetched 1200/1391  (failures so far: 3)


Daily frames: 1388  |  failures: 3


ETFs removed from candidate set: 421  (e.g. ['AFK', 'AGG', 'AGQ', 'AIA', 'ARKF', 'ARKG', 'ARKK', 'ARKQ'])



close panel: (2039, 1388)  2015-11-23 -> 2023-12-29
universe:    (2039, 1388)
members/day over 2016-01-01..2023-12-31: mean=769  min=304  max=835
distinct names ever in universe: 922

Top-10 by prev-day $vol on 2023-12-29:
AAPL     0.125
NVDA     0.110
TSLA     0.099
AMZN     0.067
COIN     0.066
AMD      0.055
MSFT     0.052
KRTX     0.049
GOOGL    0.047
SNOW     0.040
Name: prev_day_$vol_bn, dtype: float64


In [3]:
# =====================================================================
# Cell 3 — 1-min opening-range precompute (paper stop/exit rules)
#   OR = first 5 one-minute bars (09:30–09:34 ET)
#   Long:  entry stop @ OR high, stop @ OR low,  target @ entry + 10R
#   Short: entry stop @ OR low,  stop @ OR high, target @ entry - 10R
#   Else liquidate at session close.  Outputs compact daily feature panels.
# =====================================================================
from datetime import time
from concurrent.futures import ThreadPoolExecutor, as_completed

ATR_PERIOD          = 14
REL_VOL_LOOKBACK    = 14
ATR_THRESHOLD       = 0.50
PROFIT_TARGET_R     = 10
INTRADAY_WORKERS    = 6

trade_dates = close.index[(close.index >= pd.Timestamp(WINDOW_START)) &
                          (close.index <= pd.Timestamp(WINDOW_END))]
uni_tickers = sorted(universe.columns[universe.loc[trade_dates].any(axis=0)])
print(f"Universe names to pull 1-min: {len(uni_tickers)}")
print(f"Trade days: {len(trade_dates)}  ({trade_dates.min().date()} -> {trade_dates.max().date()})")


def _atr14(high: pd.Series, low: pd.Series, close_s: pd.Series) -> pd.Series:
    prev = close_s.shift(1)
    tr = pd.concat([high - low, (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)
    return tr.rolling(ATR_PERIOD, min_periods=ATR_PERIOD).mean()


def _simulate_session(day_bars: pd.DataFrame):
    """Return (strategy_return, entry_filled, direction) for one session."""
    t_open = time(9, 30)
    t_or_end = time(9, 34)
    t_entry_start = time(9, 35)

    or_bars = day_bars[(day_bars["t"] >= t_open) & (day_bars["t"] <= t_or_end)]
    if len(or_bars) < 5:
        return 0.0, False, 0

    o = or_bars.iloc[0]["Open"]
    h = or_bars["High"].max()
    l = or_bars["Low"].min()
    c = or_bars.iloc[-1]["Close"]
    if c > o:
        direction, entry, stop = 1, h, l
    elif c < o:
        direction, entry, stop = -1, l, h
    else:
        return 0.0, False, 0

    risk = abs(entry - stop)
    if risk <= 0:
        return 0.0, False, 0
    target = entry + PROFIT_TARGET_R * risk if direction == 1 else entry - PROFIT_TARGET_R * risk

    after = day_bars[day_bars["t"] >= t_entry_start]
    if after.empty:
        return 0.0, False, direction

    entered = False
    for _, row in after.iterrows():
        hi, lo, cl = row["High"], row["Low"], row["Close"]
        if not entered:
            if direction == 1 and hi >= entry:
                entered = True
                if lo <= stop:
                    return (stop - entry) / entry, True, direction
                if hi >= target:
                    return (target - entry) / entry, True, direction
            elif direction == -1 and lo <= entry:
                entered = True
                if hi >= stop:
                    return (entry - stop) / entry, True, direction
                if lo <= target:
                    return (entry - target) / entry, True, direction
        else:
            if direction == 1:
                if lo <= stop:
                    return (stop - entry) / entry, True, direction
                if hi >= target:
                    return (target - entry) / entry, True, direction
            else:
                if hi >= stop:
                    return (entry - stop) / entry, True, direction
                if lo <= target:
                    return (entry - target) / entry, True, direction

    if not entered:
        return 0.0, False, direction

    last = after.iloc[-1]["Close"]
    if direction == 1:
        return (last - entry) / entry, True, direction
    return (entry - last) / entry, True, direction


def _process_ticker(ticker: str):
    mins = hf_fetch(ticker, timeframe="1min")
    mins = mins.copy()
    mins["datetime"] = pd.to_datetime(mins["datetime"])
    mins["t"] = mins["datetime"].dt.time
    mins["date"] = mins["datetime"].dt.normalize()

    ddf = daily_frames[ticker].copy()
    ddf["datetime"] = pd.to_datetime(ddf["datetime"])
    ddf = ddf.set_index("datetime").sort_index()
    atr_s = _atr14(ddf["High"], ddf["Low"], ddf["Close"])

    rows = []
    for date, day_bars in mins.groupby("date", sort=True):
        if date not in close.index:
            continue
        or_vol = day_bars.loc[(day_bars["t"] >= time(9, 30)) &
                              (day_bars["t"] <= time(9, 34)), "Volume"].sum()
        if or_vol == 0:
            continue

        hist = mins[(mins["date"] < date) &
                    (mins["t"] >= time(9, 30)) &
                    (mins["t"] <= time(9, 34))]
        if len(hist) == 0:
            continue
        past_or_vol = hist.groupby("date")["Volume"].sum()
        past_or_vol = past_or_vol[past_or_vol.index < date].tail(REL_VOL_LOOKBACK)
        if len(past_or_vol) < REL_VOL_LOOKBACK:
            continue
        rel_vol = or_vol / past_or_vol.mean()

        strat_ret, filled, direction = _simulate_session(day_bars)
        rows.append({
            "date": date,
            "rel_vol": rel_vol,
            "atr": atr_s.get(date, np.nan),
            "or_volume": or_vol,
            "direction": direction,
            "entry_filled": filled,
            "strategy_return": strat_ret if filled else 0.0,
        })

    return ticker, pd.DataFrame(rows)


# --- parallel 1-min fetch + feature build ---
feat_parts, fetch_failures = {}, []
with ThreadPoolExecutor(max_workers=INTRADAY_WORKERS) as ex:
    futs = {ex.submit(_process_ticker, t): t for t in uni_tickers}
    for n, fut in enumerate(as_completed(futs), 1):
        t = futs[fut]
        try:
            ticker, df = fut.result()
            if df.empty:
                fetch_failures.append(t)
            else:
                feat_parts[ticker] = df.set_index("date")
        except Exception as e:
            fetch_failures.append(t)
        if n % 50 == 0:
            cached_1m = len(list(CACHE_DIR.glob("*_1min.parquet")))
            print(f"  processed {n}/{len(uni_tickers)}  |  1-min cache files: {cached_1m}", flush=True)

print(f"Tickers with features: {len(feat_parts)}  |  failures: {len(fetch_failures)}")

# --- wide panels (date x asset) aligned to close/universe ---
idx, cols = close.index, sorted(feat_parts.keys())

def _to_panel(field: str) -> pd.DataFrame:
  out = pd.DataFrame(index=idx, columns=cols, dtype=float)
  for t, df in feat_parts.items():
      if field in df.columns:
          out.loc[df.index, t] = df[field].values
  return out

rel_vol_panel        = _to_panel("rel_vol")
atr_panel            = _to_panel("atr")
or_volume_panel      = _to_panel("or_volume")
direction_panel      = _to_panel("direction")
entry_filled_panel   = _to_panel("entry_filled").fillna(0).astype(bool)
strategy_return_panel = _to_panel("strategy_return")

# persist compact features for Cell 4
feat_cache = CACHE_DIR / "orb_features_1min.parquet"
pd.concat(
    {k: _to_panel(k) for k in ["rel_vol", "atr", "or_volume", "direction",
                                "entry_filled", "strategy_return"]},
    axis=1, names=["field", "ticker"],
).to_parquet(feat_cache)

print(f"\nFeature panels: {strategy_return_panel.shape}")
print(f"rel_vol  NaN frac: {rel_vol_panel.isna().mean().mean():.1%}")
print(f"entry_filled rate (trade window): "
      f"{entry_filled_panel.loc[trade_dates].stack().mean():.3%}")
filled = strategy_return_panel.loc[trade_dates].where(entry_filled_panel.loc[trade_dates])
print(f"filled-trade return: mean={filled.stack().mean():.4f}  "
      f"median={filled.stack().median():.4f}  count={filled.stack().notna().sum():.0f}")
print(f"cached -> {feat_cache}")
strategy_return_panel.loc[trade_dates].iloc[-5:, :5]

Universe names to pull 1-min: 922
Trade days: 2012  (2016-01-04 -> 2023-12-29)
